# 08 — Yankees Analytics Case Studies: Baserunning, Defense & Roster Construction

Case studies estimating how Yankees roster construction and run-prevention choices affected team value during the 2017-2024 comparison period:

1. **"RH hitters can go oppo to the short porch"** — The roster leaned into home run production without consistently pairing it with left-handed pull power to exploit the 314-ft right field porch. The analysis tests whether lineup handedness reduced the practical value of the park dimension.
2. **"Limited emphasis on baserunning value"** — Limited emphasis stealing, fundamentally bad baserunning under Boone. Not just slow — bad decisions and no aggression on the bases.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="urllib3")

import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from fire_fishman.data.statcast import get_statcast_pitches, get_team_batting_stats
from fire_fishman.features.batted_ball import (
    classify_hit_directions,
    is_short_porch_hr,
    compute_yankee_stadium_hr_splits,
    PULL_THRESHOLD_RF,
)

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 7)

In [ ]:
# Load pitch-level Statcast data (2023-2024)
pitches_2023 = get_statcast_pitches(2023)
pitches_2024 = get_statcast_pitches(2024)
all_pitches = pd.concat([pitches_2023, pitches_2024], ignore_index=True)
print(f"Total pitches: {len(all_pitches):,}")

# Load team batting stats (2017-2024)
team_frames = []
for yr in range(2017, 2025):
    df = get_team_batting_stats(yr)
    team_frames.append(df)
team_stats = pd.concat(team_frames, ignore_index=True)
print(f"Team-seasons: {len(team_stats)}")

# Load team fielding stats (2017-2024), cached as parquet like the batting data
from fire_fishman.data.statcast import get_team_fielding_stats
fielding_frames = []
for yr in range(2017, 2025):
    tf = get_team_fielding_stats(yr)
    fielding_frames.append(tf)
all_fielding = pd.concat(fielding_frames, ignore_index=True)
print(f"Team fielding seasons: {len(all_fielding)}")

---
## Part 1: Home Run Ball Without Lefty Power

Yankee Stadium's right field porch is 314 feet — the shortest in baseball. Left-handed hitters who pull the ball are the natural exploiters of this dimension. The roster construction bet implied that right-handed hitters could create enough opposite-field power to benefit from the dimension.

The public-data concern: **opposite-field power is less common and typically lower-exit-velocity than pulled power**. LHH pull HRs are the bread and butter at Yankee Stadium. And when you stack 7 RHH in a lineup, pitchers only need one gameplan.

**Important caveat:** While LHH are more productive per PA at pulling HRs to the short RF porch (2.1x), the overall Yankee Stadium HR park factor actually favors RHH (1.24x vs 1.15x). This means the 'RH-heavy lineup' criticism is more nuanced than it first appears — the short porch helps LHH to right field, but the overall park dimensions (including the 318-ft left field) favor RHH pull power.

In [ ]:
# === Analysis 1A: LHH Pull vs RHH Oppo at Yankee Stadium ===
# All HRs at Yankee Stadium with valid hit coordinates
ys_hrs = all_pitches[
    (all_pitches["home_team"] == "NYY")
    & (all_pitches["events"] == "home_run")
    & all_pitches["hc_x"].notna()
].copy()

ys_hrs["direction"] = classify_hit_directions(ys_hrs)

# Count HRs by handedness and direction
hr_splits = (
    ys_hrs.groupby(["stand", "direction"])
    .agg(
        hr_count=("events", "count"),
        avg_exit_velo=("launch_speed", "mean"),
        avg_distance=("hit_distance_sc", "mean"),
    )
    .reset_index()
)

print("HR breakdown at Yankee Stadium (2023-2024):")
print(hr_splits.to_string(index=False))
print()

# The key comparison: LHH pull vs RHH oppo (both go to RF/short porch)
lhh_pull = hr_splits[(hr_splits["stand"] == "L") & (hr_splits["direction"] == "pull")]
rhh_oppo = hr_splits[(hr_splits["stand"] == "R") & (hr_splits["direction"] == "oppo")]

print("=" * 60)
print("THE KEY COMPARISON (both go to right field):")
print(f"  LHH pull HRs:  {lhh_pull['hr_count'].values[0]:>3}  "
      f"(avg EV {lhh_pull['avg_exit_velo'].values[0]:.1f} mph, "
      f"avg dist {lhh_pull['avg_distance'].values[0]:.0f} ft)")
print(f"  RHH oppo HRs:  {rhh_oppo['hr_count'].values[0]:>3}  "
      f"(avg EV {rhh_oppo['avg_exit_velo'].values[0]:.1f} mph, "
      f"avg dist {rhh_oppo['avg_distance'].values[0]:.0f} ft)")
print("=" * 60)

In [ ]:
# === Analysis 1B: Per-PA HR rate to RF by handedness ===
# This is the real test: controlling for plate appearances
ys_pitches = all_pitches[all_pitches["home_team"] == "NYY"]

# Count PA (approximate: events that end an at-bat)
pa_events = ys_pitches[ys_pitches["events"].notna()]

for hand, label in [("L", "LHH"), ("R", "RHH")]:
    hand_pa = len(pa_events[pa_events["stand"] == hand])
    hand_hrs_rf = len(
        ys_hrs[
            (ys_hrs["stand"] == hand)
            & (ys_hrs["hc_x"] > PULL_THRESHOLD_RF)
        ]
    )
    rate = hand_hrs_rf / hand_pa * 100 if hand_pa > 0 else 0
    print(f"{label}: {hand_hrs_rf} HRs to RF in {hand_pa:,} PA = {rate:.2f}% HR rate to RF")

print()
print("LHH produce right-field HRs at a MUCH higher per-PA rate.")
print("Every PA given to a RHH instead of a LHH at Yankee Stadium")
print("is a missed opportunity to exploit the short porch.")

In [ ]:
# === Analysis 1C: Yankees roster handedness (2017-2024) ===
# What % of Yankees PA came from RHH vs league average?

# 2023-2024 from Statcast
home_pa = all_pitches[
    (all_pitches["events"].notna())
    & (all_pitches["inning_topbot"] == "Bot")  # home team batting
].copy()

# RHH% by team at home
team_hand_data = {}
for team, grp in home_pa.groupby("home_team"):
    team_hand_data[team] = (grp["stand"] == "R").mean()

team_hand = (
    pd.DataFrame({"team": list(team_hand_data.keys()), "rhh_pct": list(team_hand_data.values())})
    .sort_values("rhh_pct", ascending=False)
    .reset_index(drop=True)
)

nyy_rhh = team_hand[team_hand["team"] == "NYY"]["rhh_pct"].values[0]
league_avg_rhh = team_hand["rhh_pct"].mean()
nyy_rank = team_hand[team_hand["team"] == "NYY"].index[0] + 1

print(f"Yankees RHH% of PA at home (2023-2024): {nyy_rhh:.1%}")
print(f"League average: {league_avg_rhh:.1%}")
print(f"Yankees rank: {nyy_rank}/{len(team_hand)} (1 = most RH-heavy)")
print()
print("Top 5 most RH-heavy lineups at home:")
for _, row in team_hand.head(5).iterrows():
    marker = " <-- SHORT PORCH" if row["team"] == "NYY" else ""
    print(f"  {row['team']}: {row['rhh_pct']:.1%}{marker}")
print()
print("Bottom 5 (most LH-heavy):")
for _, row in team_hand.tail(5).iterrows():
    print(f"  {row['team']}: {row['rhh_pct']:.1%}")

In [ ]:
# === Analysis 1D: Park factor by handedness ===
# HR rate for LHH and RHH at Yankee Stadium vs league average

bbe = all_pitches[all_pitches["events"].notna()].copy()

parks = []
for team in bbe["home_team"].unique():
    park_bbe = bbe[bbe["home_team"] == team]
    for hand in ["L", "R"]:
        hand_park = park_bbe[park_bbe["stand"] == hand]
        hand_league = bbe[bbe["stand"] == hand]
        park_hr_rate = (hand_park["events"] == "home_run").mean()
        league_hr_rate = (hand_league["events"] == "home_run").mean()
        factor = park_hr_rate / league_hr_rate if league_hr_rate > 0 else 1.0
        parks.append({"team": team, "stand": hand, "hr_rate": park_hr_rate,
                      "park_factor": factor})

park_df = pd.DataFrame(parks)
nyy_pf = park_df[park_df["team"] == "NYY"]
print("Yankee Stadium HR Park Factor (2023-2024):")
for _, row in nyy_pf.iterrows():
    label = "LHH" if row["stand"] == "L" else "RHH"
    above = "above" if row["park_factor"] > 1 else "below"
    print(f"  {label}: {row['park_factor']:.2f}x league average ({above} neutral)")

lhh_advantage = nyy_pf[nyy_pf["stand"] == "L"]["park_factor"].values[0]
rhh_factor = nyy_pf[nyy_pf["stand"] == "R"]["park_factor"].values[0]
print(f"\nLHH advantage over RHH at Yankee Stadium: {lhh_advantage - rhh_factor:.2f}x")
print("Note: The overall park factor favors RHH (1.24x vs 1.15x for LHH).")
print("The LHH advantage is specific to pull HRs to the short RF porch, not overall HR production.")

In [ ]:
# === Analysis 1E: Pitchers get comfortable vs all-RH lineup ===
# When 7+ RHH are in the lineup, pitchers can use one gameplan.
# LHH force pitchers to adjust — different pitch mix, different sequencing.

# Pitch mix to Yankees RHH vs LHH at home
nyy_home_pitches = all_pitches[
    (all_pitches["home_team"] == "NYY")
    & (all_pitches["inning_topbot"] == "Bot")
    & all_pitches["pitch_type"].notna()
].copy()

PITCH_GROUPS = {
    "Fastball": ["FF", "SI", "FC"],
    "Breaking": ["SL", "CU", "KC", "SV", "SC", "ST"],
    "Offspeed": ["CH", "FS", "FO"],
}

def get_pitch_group(pt):
    for group, codes in PITCH_GROUPS.items():
        if pt in codes:
            return group
    return "Other"

nyy_home_pitches["pitch_group"] = nyy_home_pitches["pitch_type"].apply(get_pitch_group)

print("Pitch mix to Yankees batters at home (2023-2024):")
print("=" * 50)
for hand, label in [("R", "RHH"), ("L", "LHH")]:
    h = nyy_home_pitches[nyy_home_pitches["stand"] == hand]
    mix = h["pitch_group"].value_counts(normalize=True)
    print(f"  {label}: Fastball {mix.get('Fastball', 0):.1%}  "
          f"Breaking {mix.get('Breaking', 0):.1%}  "
          f"Offspeed {mix.get('Offspeed', 0):.1%}")

print()
print("When your lineup is 7 RHH, pitchers see the same handedness")
print("batter after batter. No adjustment needed. One gameplan.")
print("LHH force pitchers to flip their sequencing, use different")
print("pitch tunnels, and adjust release angles. Lineup diversity")
print("is a competitive advantage the Yankees gave away for free.")

In [ ]:
# === Analysis 1F: Compare to all 30 parks ===
# Which parks have the biggest LHH HR advantage? Yankee Stadium should be near the top.

park_advantage = (
    park_df.pivot(index="team", columns="stand", values="park_factor")
    .reset_index()
)
park_advantage.columns = ["team", "LHH_factor", "RHH_factor"]
park_advantage["LHH_advantage"] = park_advantage["LHH_factor"] - park_advantage["RHH_factor"]
park_advantage = park_advantage.sort_values("LHH_advantage", ascending=False)

print("Parks ranked by LHH HR advantage over RHH (2023-2024):")
print("(Positive = park favors LHH more than RHH)")
print()
for i, (_, row) in enumerate(park_advantage.iterrows()):
    marker = " <<<" if row["team"] == "NYY" else ""
    print(f"  {i+1:2d}. {row['team']}: LHH {row['LHH_factor']:.2f}x, "
          f"RHH {row['RHH_factor']:.2f}x, advantage {row['LHH_advantage']:+.2f}{marker}")

In [ ]:
# === FIGURE 1: Short Porch Handedness Analysis ===

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Panel A: Spray chart of HRs at Yankee Stadium ---
ax = axes[0]
# LHH pull HRs (to RF)
lhh = ys_hrs[ys_hrs["stand"] == "L"]
lhh_pull_mask = lhh["hc_x"] > PULL_THRESHOLD_RF
ax.scatter(lhh.loc[~lhh_pull_mask, "hc_x"], lhh.loc[~lhh_pull_mask, "hc_y"],
           s=30, alpha=0.4, color="#95a5a6", label="LHH other", zorder=3)
ax.scatter(lhh.loc[lhh_pull_mask, "hc_x"], lhh.loc[lhh_pull_mask, "hc_y"],
           s=60, alpha=0.7, color="#2ecc71", label="LHH pull (RF)", zorder=5)

# RHH oppo HRs (to RF)
rhh = ys_hrs[ys_hrs["stand"] == "R"]
rhh_oppo_mask = rhh["hc_x"] > PULL_THRESHOLD_RF
ax.scatter(rhh.loc[~rhh_oppo_mask, "hc_x"], rhh.loc[~rhh_oppo_mask, "hc_y"],
           s=30, alpha=0.4, color="#bdc3c7", label="RHH other", zorder=3)
ax.scatter(rhh.loc[rhh_oppo_mask, "hc_x"], rhh.loc[rhh_oppo_mask, "hc_y"],
           s=60, alpha=0.7, color="#e74c3c", label="RHH oppo (RF)", zorder=5)

# Short porch zone
ax.axvline(x=PULL_THRESHOLD_RF, color="#e67e22", linestyle="--", alpha=0.5, label="RF zone")
ax.set_xlabel("hc_x (higher = right field)")
ax.set_ylabel("hc_y")
ax.set_title("Yankee Stadium HRs (2023-24)\nSpray Chart", fontweight="bold")
ax.legend(fontsize=8, loc="upper left")
ax.invert_yaxis()

# --- Panel B: HR count to RF by handedness ---
ax = axes[1]
lhh_pull_count = lhh_pull_mask.sum()
rhh_oppo_count = rhh_oppo_mask.sum()
bars = ax.bar(["LHH Pull\n(natural)", "RHH Oppo\n(forced)"],
              [lhh_pull_count, rhh_oppo_count],
              color=["#2ecc71", "#e74c3c"], edgecolor="white", width=0.6)
ax.bar_label(bars, fontsize=14, fontweight="bold", padding=5)
ax.set_ylabel("Home Runs to Right Field")
ax.set_title("HRs to Short Porch\nLHH Pull vs RHH Oppo", fontweight="bold")

# --- Panel C: Park factor by handedness ---
ax = axes[2]
nyy_lhh_pf = park_df[(park_df["team"] == "NYY") & (park_df["stand"] == "L")]["park_factor"].values[0]
nyy_rhh_pf = park_df[(park_df["team"] == "NYY") & (park_df["stand"] == "R")]["park_factor"].values[0]
bars = ax.bar(["LHH", "RHH"], [nyy_lhh_pf, nyy_rhh_pf],
              color=["#2ecc71", "#e74c3c"], edgecolor="white", width=0.5)
ax.bar_label(bars, fmt="%.2fx", fontsize=14, fontweight="bold", padding=5)
ax.axhline(y=1.0, color="#7f8c8d", linestyle="--", alpha=0.7, label="League avg (1.0x)")
ax.set_ylabel("HR Park Factor")
ax.set_title("Yankee Stadium HR\nPark Factor by Handedness", fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, max(nyy_lhh_pf, nyy_rhh_pf) * 1.3)

plt.tight_layout()
plt.savefig("../outputs/figures/short_porch_handedness.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/figures/short_porch_handedness.png")

---
## Part 2: The Baserunning Decline

Under Boone and Yankees analytics, the Yankees abandoned baserunning as a competitive tool. They stole fewer bases, took fewer extra bases, and made terrible decisions on the bases. The numbers tell the story: they went from 7th in BsR (2017) to **30th** (2024).

This isn't just about speed — it's about philosophy. Fast teams with aggressive baserunning force errors, create extra runs, and put pressure on pitchers. The Yankees instead played station-to-station, waiting for a home run that increasingly wasn't coming.

In [ ]:
# === Analysis 2A: Yankees BsR collapse (2017-2024) ===
nyy_stats = team_stats[team_stats["Team"] == "NYY"].sort_values("Season")

print("Yankees Baserunning (2017-2024):")
print("=" * 70)
print(f"{'Year':>6} {'BsR':>7} {'Rank':>6} {'SB':>5} {'CS':>5} {'UBR':>7} {'wSB':>7} {'Spd':>5}")
print("-" * 70)

for _, row in nyy_stats.iterrows():
    yr = int(row["Season"])
    yr_all = team_stats[team_stats["Season"] == yr].sort_values("BsR", ascending=False).reset_index(drop=True)
    rank = yr_all[yr_all["Team"] == "NYY"].index[0] + 1
    print(f"{yr:>6} {row['BsR']:>7.1f} {rank:>4}/30 {int(row['SB']):>5} {int(row['CS']):>5} "
          f"{row['UBR']:>7.1f} {row['wSB']:>7.1f} {row['Spd']:>5.1f}")

print("=" * 70)
print()
print("BsR = total baserunning runs above average")
print("UBR = extra base taking runs (advancing on hits, fly balls)")
print("wSB = stolen base runs (SB value minus CS cost)")
print()
total_bsr = nyy_stats["BsR"].sum()
total_ubr = nyy_stats["UBR"].sum()
print(f"TOTAL BsR (2017-2024): {total_bsr:.1f} runs")
print(f"TOTAL UBR (2017-2024): {total_ubr:.1f} runs")
print(f"That's roughly {abs(total_bsr) / 10:.1f} wins lost to bad baserunning.")

In [ ]:
# === Analysis 2B: Yankees vs best baserunning teams ===
# Average BsR 2018-2022 (the Yankees comparison period of bad baserunning)
comparison_period = team_stats[team_stats["Season"].between(2018, 2022)]

era_avg = (
    comparison_period.groupby("Team")["BsR"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
era_avg.columns = ["Team", "avg_BsR"]

print("Average BsR by team (2018-2022, Yankees comparison period):")
print("=" * 45)
for i, (_, row) in enumerate(era_avg.iterrows()):
    marker = " <<<" if row["Team"] == "NYY" else ""
    print(f"  {i+1:2d}. {row['Team']}: {row['avg_BsR']:>+6.1f} runs/yr{marker}")

nyy_avg = era_avg[era_avg["Team"] == "NYY"]["avg_BsR"].values[0]
best_avg = era_avg.iloc[0]["avg_BsR"]
best_team = era_avg.iloc[0]["Team"]
print(f"\nGap between Yankees ({nyy_avg:+.1f}) and {best_team} ({best_avg:+.1f}): "
      f"{best_avg - nyy_avg:.1f} runs/yr = ~{(best_avg - nyy_avg) / 10:.1f} wins/yr")

In [ ]:
# === Analysis 2C: UBR collapse — not just stealing, bad fundamentals ===
# UBR captures advancing on hits, fly balls, passed balls — extra-base advancement component
# wSB captures just the steal game. UBR is where Boone's teams really fall apart.

print("Yankees UBR vs wSB (2017-2024):")
print("UBR = baserunning decisions (extra bases, advancement)")
print("wSB = stolen base value")
print("=" * 50)
for _, row in nyy_stats.iterrows():
    yr = int(row["Season"])
    print(f"  {yr}: UBR {row['UBR']:>+6.1f}  wSB {row['wSB']:>+5.1f}  "
          f"BsR {row['BsR']:>+6.1f}")
print()
print("The UBR collapse is WORSE than the stolen base issue.")
print("This isn't just 'we don't steal' — the team can't run the bases.")
print("Outs on bases, limited advancement on balls in dirt, missed")
print("opportunities to go first-to-third. Fundamental baserunning.")

In [ ]:
# === Analysis 2D: Stolen base attempts — they just didn't bother ===

# SB totals and league rank by year
print("Yankees Stolen Bases vs League (2017-2024):")
print("=" * 60)
for yr in range(2017, 2025):
    yr_df = team_stats[team_stats["Season"] == yr].sort_values("SB", ascending=False).reset_index(drop=True)
    nyy_row = yr_df[yr_df["Team"] == "NYY"]
    rank = nyy_row.index[0] + 1
    sb = int(nyy_row["SB"].values[0])
    league_avg = yr_df["SB"].mean()
    leader = yr_df.iloc[0]
    print(f"  {yr}: {sb:>3} SB (rank {rank:>2}/30, league avg {league_avg:.0f}, "
          f"leader: {leader['Team']} {int(leader['SB'])}")

print()
# 2019-2021 was the nadir
nadir = team_stats[
    (team_stats["Team"] == "NYY") & (team_stats["Season"].between(2019, 2021))
]
print(f"2019-2021 peak Yankees comparison period: {nadir['SB'].mean():.0f} SB/yr average")
print("They literally stopped running.")

In [ ]:
# === Analysis 2E: Age profile — old and immobile ===

print("Team Average Age (weighted by PA) — Yankees vs League (2017-2024):")
print("=" * 55)
for yr in range(2017, 2025):
    yr_df = team_stats[team_stats["Season"] == yr]
    nyy_age = yr_df[yr_df["Team"] == "NYY"]["Age"].values[0]
    league_age = yr_df["Age"].mean()
    rank = yr_df.sort_values("Age", ascending=False).reset_index(drop=True)
    nyy_rank = rank[rank["Team"] == "NYY"].index[0] + 1
    diff = nyy_age - league_age
    print(f"  {yr}: NYY {nyy_age:.1f} vs league {league_age:.1f} "
          f"({diff:+.1f}, rank {nyy_rank}/30 oldest)")

print()
# Correlation: age vs BsR
corr = team_stats[["Age", "BsR"]].corr().iloc[0, 1]
print(f"Correlation between team age and BsR (2017-2024): r = {corr:.2f}")
print("Older teams run worse. The Yankees chose to be both old AND bad at running.")

In [ ]:
# === Analysis 2F: HR dependence — home run or bust ===
# What % of Yankees runs came via HR vs league?

print("HR Dependence: Yankees vs League (2017-2024):")
print("(HR as % of total runs scored)")
print("=" * 55)
for yr in range(2017, 2025):
    yr_df = team_stats[team_stats["Season"] == yr]
    yr_df = yr_df.copy()
    yr_df["hr_pct_of_runs"] = yr_df["HR"] / yr_df["R"]
    nyy_pct = yr_df[yr_df["Team"] == "NYY"]["hr_pct_of_runs"].values[0]
    league_pct = yr_df["hr_pct_of_runs"].mean()
    rank = yr_df.sort_values("hr_pct_of_runs", ascending=False).reset_index(drop=True)
    nyy_rank = rank[rank["Team"] == "NYY"].index[0] + 1
    print(f"  {yr}: NYY {nyy_pct:.1%} vs league {league_pct:.1%} "
          f"(rank {nyy_rank}/30 most HR-dependent)")

print()
print("When you don't steal, don't take extra bases, and don't have")
print("lefty power to exploit your own park — you're left waiting")
print("for home runs. When they don't come (postseason, good pitching),")
print("there's no Plan B.")

In [ ]:
# === FIGURE 2: Baserunning Decline ===

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Panel A: BsR by year (Yankees highlighted) ---
ax = axes[0]
years = list(range(2017, 2025))
nyy_bsr = [nyy_stats[nyy_stats["Season"] == yr]["BsR"].values[0] for yr in years]
# League average
league_bsr = [team_stats[team_stats["Season"] == yr]["BsR"].mean() for yr in years]
# Best team each year
best_bsr = [team_stats[team_stats["Season"] == yr]["BsR"].max() for yr in years]

ax.plot(years, nyy_bsr, "o-", color="#e74c3c", linewidth=2.5, markersize=8, label="Yankees", zorder=5)
ax.plot(years, league_bsr, "s--", color="#7f8c8d", linewidth=1.5, markersize=5, label="League avg", zorder=3)
ax.plot(years, best_bsr, "^--", color="#2ecc71", linewidth=1.5, markersize=5, label="Best team", zorder=3)
ax.axhline(y=0, color="black", linewidth=0.5, alpha=0.3)
ax.fill_between(years, nyy_bsr, 0, alpha=0.15, color="#e74c3c")
ax.set_xlabel("Season")
ax.set_ylabel("Baserunning Runs (BsR)")
ax.set_title("Yankees BsR Collapse\n2017-2024", fontweight="bold")
ax.legend(fontsize=9)

# --- Panel B: BsR breakdown (UBR vs wSB) ---
ax = axes[1]
nyy_ubr = [nyy_stats[nyy_stats["Season"] == yr]["UBR"].values[0] for yr in years]
nyy_wsb = [nyy_stats[nyy_stats["Season"] == yr]["wSB"].values[0] for yr in years]

x = np.arange(len(years))
width = 0.35
bars1 = ax.bar(x - width / 2, nyy_ubr, width, label="UBR (baserunning IQ)",
               color="#e74c3c", edgecolor="white")
bars2 = ax.bar(x + width / 2, nyy_wsb, width, label="wSB (steal value)",
               color="#e67e22", edgecolor="white")
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45)
ax.set_ylabel("Runs Above Average")
ax.set_title("UBR vs wSB\nBad Running > Bad Stealing", fontweight="bold")
ax.legend(fontsize=9)

# --- Panel C: Team BsR rankings (2018-2022 avg, horizontal bar) ---
ax = axes[2]
era_avg_sorted = era_avg.sort_values("avg_BsR", ascending=True)
colors = ["#e74c3c" if t == "NYY" else "#3498db" for t in era_avg_sorted["Team"]]
ax.barh(range(len(era_avg_sorted)), era_avg_sorted["avg_BsR"],
        color=colors, edgecolor="white", height=0.7)
ax.set_yticks(range(len(era_avg_sorted)))
ax.set_yticklabels(era_avg_sorted["Team"], fontsize=8)
ax.axvline(x=0, color="black", linewidth=0.5)
ax.set_xlabel("Avg Baserunning Runs (BsR)")
ax.set_title("Team BsR Rankings\n2018-2022 Average", fontweight="bold")

plt.tight_layout()
plt.savefig("../outputs/figures/baserunning_decline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/figures/baserunning_decline.png")

---
## Part 4: Defensive Decline and Postseason Context

The same analytics department that underweighted baserunning, lineup balance, and defense. From 2018-2021, the Yankees had the **2nd worst OAA (Outs Above Average) in baseball** — near the bottom of the league. The roster profile leaned heavily toward offense-first bats.

Meanwhile, the teams that beat them in October — Houston, Tampa Bay, LA — were all top-6 defensive teams. Defense doesn't just save runs; it saves games in high-leverage situations when every out matters. The Yankees built a roster with no Plan B: no speed, no defense, no lineup diversity. Just wait for the home run. And when it doesn't come in October against elite pitching, there's nothing left.

In [ ]:
# === Analysis 4A: Yankees defensive collapse (2017-2024) ===
nyy_fielding = all_fielding[all_fielding["Team"] == "Yankees"].sort_values("Season")

print("Yankees Defense (2017-2024):")
print("=" * 75)
print(f"{'Year':>6} {'DRS':>6} {'OAA':>6} {'UZR':>7} {'Def':>7} {'Rank(DRS)':>10}")
print("-" * 75)

for _, row in nyy_fielding.iterrows():
    yr = int(row["Season"])
    yr_all = all_fielding[all_fielding["Season"] == yr].sort_values("DRS", ascending=False).reset_index(drop=True)
    rank = yr_all[yr_all["Team"] == "Yankees"].index[0] + 1
    oaa = row.get("OAA", "N/A")
    oaa_str = f"{oaa:>+5.0f}" if isinstance(oaa, (int, float)) else "  N/A"
    print(f"{yr:>6} {row['DRS']:>+5.0f} {oaa_str} {row['UZR']:>+7.1f} {row['Def']:>+7.1f} {rank:>7}/30")

print("=" * 75)
print()

# The bad years
bad = nyy_fielding[nyy_fielding["Season"].between(2018, 2021)]
print(f"2018-2021 DAMAGE:")
print(f"  Total DRS: {bad['DRS'].sum():+.0f}")
print(f"  Total OAA: {bad['OAA'].sum():+.0f}")
print(f"  Total Def: {bad['Def'].sum():+.1f} runs")
print(f"  That's ~{abs(bad['Def'].sum()) / 10:.1f} wins lost to bad defense")
print()
print("Then 2022 happened: IKF at SS, Trevino catching, and suddenly")
print("they're #1 in DRS (+124). Proving the talent to play defense")
print("existed — they just chose not to prioritize it for 4 years.")

In [ ]:
# === Analysis 4B: The teams that beat them in October ===
# The Yankees lost in the postseason to teams that excelled at
# exactly the things were underweighted in the roster profile.

october_opponents = {
    2017: ("Astros", "ALCS L 3-4", "Elite defense"),
    2018: ("Red Sox", "ALDS L 1-3", "Better all-around team"),
    2019: ("Astros", "ALCS L 2-4", "Elite defense + baserunning"),
    2020: ("Rays", "ALDS L 2-3", "Speed, defense, versatility"),
    2021: ("Red Sox", "ALWC L", "Wild card loss"),
    2022: ("Astros", "ALCS L 0-4", "Houston: #1 DRS"),
    2023: ("--", "Missed playoffs", "84-78, missed postseason"),
    2024: ("Dodgers", "WS L 1-4", "Strong two-way roster"),
}

print("Yankees October Results vs Team Quality (2017-2024):")
print("=" * 80)

for yr, (opp, result, note) in october_opponents.items():
    yr_df = all_fielding[all_fielding["Season"] == yr]
    
    nyy_def = yr_df[yr_df["Team"] == "Yankees"]["Def"].values[0]
    
    if opp != "--":
        opp_row = yr_df[yr_df["Team"] == opp]
        if len(opp_row) > 0:
            opp_def = opp_row["Def"].values[0]
            opp_drs = opp_row["DRS"].values[0]
            print(f"  {yr}: {result:20s} NYY Def {nyy_def:>+6.1f} vs {opp} Def {opp_def:>+6.1f}  ({note})")
        else:
            print(f"  {yr}: {result:20s} NYY Def {nyy_def:>+6.1f}  ({note})")
    else:
        print(f"  {yr}: {result:20s} NYY Def {nyy_def:>+6.1f}  ({note})")

print("=" * 80)
print()
print("The pattern is clear: the Yankees lose in October to teams that")
print("play complete baseball — defense, speed, lineup diversity.")
print("You can't out-homer elite pitchers for 7 games. When the HRs")
print("stop, you need other ways to win. The Fishman Yankees had none.")

In [ ]:
# === Analysis 4C: Composite value deficit ===
# Summarizing pressure performance, baserunning, and defense.
# FanGraphs Clutch metric measures how much a team over/underperforms
# their expected WPA in high-leverage situations.

# Also: WPA (Win Probability Added), pLI (avg leverage), Clutch
nyy_batting = team_stats[team_stats["Team"] == "NYY"].sort_values("Season")

print("The Team Value Composite: Yankees Clutch & Aggressiveness (2017-2024):")
print("=" * 70)
print(f"{'Year':>6} {'Clutch':>8} {'BsR':>7} {'Def':>7} {'Spd':>5}  {'Diagnosis'}")
print("-" * 70)

for _, row in nyy_batting.iterrows():
    yr = int(row["Season"])
    clutch = row.get("Clutch", 0)
    bsr = row["BsR"]
    spd = row["Spd"]
    
    # Get Def from fielding data
    nyy_f = all_fielding[(all_fielding["Team"] == "Yankees") & (all_fielding["Season"] == yr)]
    def_val = nyy_f["Def"].values[0] if len(nyy_f) > 0 else 0
    
    # Value Composite score: combination of aggressiveness metrics
    # Clutch (pressure performance) + BsR (baserunning aggression) + Def (effort/positioning)
    value_composite = clutch + bsr / 5 + def_val / 20
    
    if value_composite > 2:
        diag = "Has value_composite"
    elif value_composite > 0:
        diag = "Some value_composite"
    elif value_composite > -2:
        diag = "Low value_composite"
    else:
        diag = "LOW COMPOSITE"
    
    print(f"{yr:>6} {clutch:>+8.1f} {bsr:>+7.1f} {def_val:>+7.1f} {spd:>5.1f}  {diag}")

print("=" * 70)
print()
print("VALUE COMPOSITE FORMULA:")
print("  Value Composite = Clutch + (BsR / 5) + (Def / 20)")
print("  > 2.0: Has value_composite (plays winning baseball)")
print("  0-2:   Some value_composite (mediocre intensity)")
print("  < 0:   Low value_composite (going through the motions)")
print("  < -2:  LOW COMPOSITE (low non-offensive value)")
print()
print("Value Composite measures what analytics can't buy: aggressiveness on the")
print("bases, effort on defense, and performance when it matters most.")
print("The Fishman Yankees optimized for one thing (power) and let")
print("everything else atrophy. You can't build a championship team")
print("on spreadsheets alone — you need players who play with urgency,")
print("take the extra base, make the diving play, and step up in October.")

In [ ]:
# === FIGURE 3: Defensive & Value Composite Analysis ===

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Panel A: Def runs by year ---
ax = axes[0]
years = list(range(2017, 2025))
nyy_def = [all_fielding[(all_fielding["Team"] == "Yankees") & (all_fielding["Season"] == yr)]["Def"].values[0] for yr in years]
nyy_oaa = [all_fielding[(all_fielding["Team"] == "Yankees") & (all_fielding["Season"] == yr)]["OAA"].values[0] for yr in years]

ax.bar(years, nyy_def, color=["#2ecc71" if d > 0 else "#e74c3c" for d in nyy_def],
       edgecolor="white", width=0.7)
ax.plot(years, nyy_oaa, "D-", color="#3498db", linewidth=2, markersize=6, label="OAA", zorder=5)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_xlabel("Season")
ax.set_ylabel("Runs / Outs")
ax.set_title("Yankees Def Runs & OAA\n2017-2024", fontweight="bold")
ax.legend(fontsize=9)

# --- Panel B: Value Composite score by year ---
ax = axes[1]
value_composite_scores = []
for yr in years:
    yr_bat = team_stats[(team_stats["Team"] == "NYY") & (team_stats["Season"] == yr)]
    yr_fld = all_fielding[(all_fielding["Team"] == "Yankees") & (all_fielding["Season"] == yr)]
    clutch = yr_bat["Clutch"].values[0] if len(yr_bat) > 0 else 0
    bsr = yr_bat["BsR"].values[0] if len(yr_bat) > 0 else 0
    def_val = yr_fld["Def"].values[0] if len(yr_fld) > 0 else 0
    value_composite = clutch + bsr / 5 + def_val / 20
    value_composite_scores.append(value_composite)

colors = ["#2ecc71" if d > 0 else "#e74c3c" for d in value_composite_scores]
ax.bar(years, value_composite_scores, color=colors, edgecolor="white", width=0.7)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.axhline(y=2, color="#2ecc71", linestyle="--", alpha=0.5, label='"Has value_composite" threshold')
ax.axhline(y=-2, color="#e74c3c", linestyle="--", alpha=0.5, label='"No value_composite" threshold')
ax.set_xlabel("Season")
ax.set_ylabel("Value Composite Score")
ax.set_title("Yankees Value Composite Score\nClutch + BsR + Def", fontweight="bold")
ax.legend(fontsize=8)

# --- Panel C: Yankees vs October opponents (Def comparison) ---
ax = axes[2]
oct_years = [2017, 2018, 2019, 2020, 2021, 2022, 2024]
oct_opps = ["Astros", "Red Sox", "Astros", "Rays", "Red Sox", "Astros", "Dodgers"]
nyy_defs = []
opp_defs = []
for yr, opp in zip(oct_years, oct_opps):
    n = all_fielding[(all_fielding["Team"] == "Yankees") & (all_fielding["Season"] == yr)]["Def"].values[0]
    o_row = all_fielding[(all_fielding["Team"] == opp) & (all_fielding["Season"] == yr)]
    o = o_row["Def"].values[0] if len(o_row) > 0 else 0
    nyy_defs.append(n)
    opp_defs.append(o)

x = np.arange(len(oct_years))
width = 0.35
ax.bar(x - width/2, nyy_defs, width, label="Yankees", color="#e74c3c", edgecolor="white")
ax.bar(x + width/2, opp_defs, width, label="Opponent", color="#2ecc71", edgecolor="white")
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f"{yr}\nvs {opp}" for yr, opp in zip(oct_years, oct_opps)],
                    fontsize=7, rotation=45, ha="right")
ax.set_ylabel("Defensive Runs (Def)")
ax.set_title("Yankees vs October Opponents\nDefensive Comparison", fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../outputs/figures/defense_and_value_composite.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/figures/defense_and_value_composite.png")

---
## Part 5: The 2022 Paradox — Why 99 Wins Did Not Translate in October

The 2022 Yankees are the main counterexample. They won 99 games, had the **#1 DRS in baseball (+124)**, and ranked in the top 6 in team value composite. If the thesis is that Yankees analytics underweighted defense, baserunning, and non-offensive value, 2022 needs separate interpretation.

No. 2022 is actually the strongest evidence *for* the thesis. The Yankees fixed defense (Grit) and showed up in clutch spots (Pressure), but **never fixed the baserunning (Hustle)**. That single missing component made them a team with no Plan B when the home runs stopped — which is exactly what happened in the ALCS.

In [ ]:
# === Analysis 5A: Component Imbalance — Value Composite only works when balanced ===
# Same definition as notebook 09, computed by the shared src module.
# (An earlier revision re-implemented the composite here with its own team-name
# map and ad-hoc 0.30/0.35/0.35 weights; the map silently dropped the Rays and
# White Sox from the merge.)
from fire_fishman.features.team_value import compute_value_composite, merge_team_batting_fielding

value_composite_df = compute_value_composite(
    merge_team_batting_fielding(team_stats, all_fielding)
)
assert value_composite_df["OAA"].notna().all(), "merge dropped OAA for some team-season"
print(f"Composite computed for {len(value_composite_df)} team-seasons\n")

# Find top-10 Value Composite teams each year, check for component imbalance
print("TOP-10 VALUE COMPOSITE TEAMS WITH A COMPONENT BELOW -0.50")
print("=" * 95)
print(f"{'Year':>4}  {'Team':>4}  {'Rank':>4}  {'Pressure':>9} {'Hustle':>8} {'Grit':>6} {'Value':>7}  {'Weak spot'}")
print("-" * 95)

imbalanced_count = 0
balanced_count = 0
top10_rows = []
for yr in range(2017, 2025):
    season = value_composite_df[value_composite_df['Season'] == yr].sort_values('value_composite', ascending=False).reset_index(drop=True)
    top10 = season.head(10)
    top10_rows.append(top10)
    for idx, row in top10.iterrows():
        rank = idx + 1
        components = {'Pressure': row['pressure'], 'Hustle': row['hustle'], 'Grit': row['grit']}
        weak = [f"{k} ({v:+.2f})" for k, v in components.items() if v < -0.50]
        if weak:
            imbalanced_count += 1
            marker = " <<<" if row['Team'] == 'NYY' else ""
            print(f"{yr:>4}  {row['Team']:>4}  {rank:>3}   {row['pressure']:>+9.2f} {row['hustle']:>+8.2f} "
                  f"{row['grit']:>+6.2f} {row['value_composite']:>+7.2f}  {', '.join(weak)}{marker}")
        else:
            balanced_count += 1

top10_df = pd.concat(top10_rows, ignore_index=True)
print("-" * 95)
print(f"\nOf {balanced_count + imbalanced_count} top-10 Value Composite team-seasons:")
print(f"  Balanced (all components > -0.50): {balanced_count}")
print(f"  Imbalanced (any component < -0.50): {imbalanced_count}")
print()

# 2022 ALCS head-to-head
season_2022 = value_composite_df[value_composite_df['Season'] == 2022]
nyy_2022 = season_2022[season_2022['Team'] == 'NYY'].iloc[0]
hou_2022 = season_2022[season_2022['Team'] == 'HOU'].iloc[0]

print("2022 ALCS HEAD-TO-HEAD: VALUE COMPOSITE COMPONENTS")
print("=" * 65)
print(f"{'':>12} {'Pressure':>10} {'Hustle':>10} {'Grit':>10} {'Value':>10}")
print("-" * 65)
print(f"{'Yankees':>12} {nyy_2022['pressure']:>+10.2f} {nyy_2022['hustle']:>+10.2f} "
      f"{nyy_2022['grit']:>+10.2f} {nyy_2022['value_composite']:>+10.2f}")
print(f"{'Astros':>12} {hou_2022['pressure']:>+10.2f} {hou_2022['hustle']:>+10.2f} "
      f"{hou_2022['grit']:>+10.2f} {hou_2022['value_composite']:>+10.2f}")
print("-" * 65)

nyy_rank_2022 = int(season_2022.sort_values('value_composite', ascending=False)
                    .reset_index(drop=True).query("Team == 'NYY'").index[0]) + 1
hou_grit_rank = int((season_2022['grit'] > hou_2022['grit']).sum()) + 1
print(f"\n2022 Yankees: composite rank #{nyy_rank_2022} of 30; Hustle {nyy_2022['hustle']:+.2f} "
      f"(min Hustle among all top-10 team-seasons: {top10_df['hustle'].min():+.2f}).")
print(f"2022 Astros: Grit {hou_2022['grit']:+.2f}, ranked #{hou_grit_rank} of 30 that season.")
print("\nWhen Houston's pitching shut down the HR, the Yankees had no way to manufacture runs.")


In [ ]:
# === Analysis 5B: The Second-Half Collapse ===
# 2022 Yankees: 64-28 first half (.696), 35-35 second half (.500)
# The most famous collapse in recent memory

print("2022 YANKEES: THE TALE OF TWO HALVES")
print("=" * 70)
print()
print(f"{'Period':<20} {'W-L':>8} {'Win%':>8} {'Pace':>12}")
print("-" * 70)
print(f"{'First Half':<20} {'64-28':>8} {'.696':>8} {'117-win':>12}")
print(f"{'Second Half':<20} {'35-35':>8} {'.500':>8} {'81-win':>12}")
print(f"{'Full Season':<20} {'99-63':>8} {'.611':>8} {'':>12}")
print("-" * 70)
print()

# The HR-dependency angle: Yankees HR% of runs
print("HR-DEPENDENCY: % OF RUNS SCORED VIA HOME RUN (2022)")
print("=" * 70)
yr_stats = value_composite_df[value_composite_df['Season'] == 2022].copy()
yr_stats['HR_pct'] = yr_stats['HR'] / yr_stats['R'] * 100
yr_stats = yr_stats.sort_values('HR_pct', ascending=False).reset_index(drop=True)
print(f"{'Rank':>4}  {'Team':>4}  {'HR':>5}  {'R':>5}  {'HR%':>6}")
print("-" * 35)
for idx, row in yr_stats.head(10).iterrows():
    marker = " <<<" if row['Team'] == 'NYY' else ""
    print(f"{idx+1:>4}  {row['Team']:>4}  {int(row['HR']):>5}  {int(row['R']):>5}  {row['HR_pct']:>5.1f}%{marker}")

nyy_rank = yr_stats[yr_stats['Team'] == 'NYY'].index[0] + 1
hou_rank = yr_stats[yr_stats['Team'] == 'HOU'].index[0] + 1
print(f"\nYankees: #{nyy_rank} most HR-dependent")
print(f"Astros: #{hou_rank} most HR-dependent")
print()

# HR-dependency rank over time
print("YANKEES HR-DEPENDENCY RANK (2017-2024)")
print("=" * 50)
for yr in range(2017, 2025):
    yr_s = value_composite_df[value_composite_df['Season'] == yr].copy()
    yr_s['HR_pct'] = yr_s['HR'] / yr_s['R'] * 100
    yr_s = yr_s.sort_values('HR_pct', ascending=False).reset_index(drop=True)
    nyy_r = yr_s[yr_s['Team'] == 'NYY'].index[0] + 1
    nyy_pct = yr_s[yr_s['Team'] == 'NYY']['HR_pct'].values[0]
    flag = " <<<" if nyy_r <= 3 else ""
    print(f"  {yr}: #{nyy_r:>2} ({nyy_pct:.1f}%){flag}")
print()
print("When you're top-3 in HR-dependency and your Hustle is -0.98,")
print("there is literally no Plan B when elite pitching suppresses the long ball.")

In [ ]:
# === Analysis 5C: The ALCS Breakdown ===
# In the 2022 ALCS, the Yankees scored 2, 3, 0, 2 runs (1.75 R/G) vs Houston
# Houston scored 4, 3, 5, 6 runs (4.5 R/G)

alcs_games = pd.DataFrame({
    'Game': [1, 2, 3, 4],
    'NYY_R': [2, 3, 0, 2],
    'HOU_R': [4, 3, 5, 6],
    'NYY_HR': [1, 2, 0, 1],
    'HOU_HR': [1, 1, 1, 0],
})

print("2022 ALCS GAME LOG")
print("=" * 65)
print(f"{'Game':>4}  {'NYY R':>6}  {'HOU R':>6}  {'NYY HR':>7}  {'HOU HR':>7}")
print("-" * 65)
for _, g in alcs_games.iterrows():
    print(f"  {int(g['Game']):>2}  {int(g['NYY_R']):>6}  {int(g['HOU_R']):>6}  "
          f"{int(g['NYY_HR']):>7}  {int(g['HOU_HR']):>7}")
print("-" * 65)
nyy_total_r = alcs_games['NYY_R'].sum()
hou_total_r = alcs_games['HOU_R'].sum()
nyy_total_hr = alcs_games['NYY_HR'].sum()
hou_total_hr = alcs_games['HOU_HR'].sum()
nyy_rpg = nyy_total_r / 4
hou_rpg = hou_total_r / 4
print(f"{'Total':>6}  {nyy_total_r:>6}  {hou_total_r:>6}  {nyy_total_hr:>7}  {hou_total_hr:>7}")
print(f"{'R/G':>6}  {nyy_rpg:>6.1f}  {hou_rpg:>6.1f}")
print()

# The manufacturing gap
nyy_non_hr_runs = nyy_total_r - nyy_total_hr  # rough approximation
hou_non_hr_runs = hou_total_r - hou_total_hr
print("THE MANUFACTURING GAP")
print("-" * 50)
print(f"  Yankees runs via HR: ~{nyy_total_hr * 1} (HRs were mostly solo)")
print(f"  Yankees other runs:  ~{nyy_non_hr_runs}")
print(f"  Astros runs via HR:  ~{hou_total_hr * 1}")
print(f"  Astros other runs:   ~{hou_non_hr_runs}")
print()
print(f"Houston manufactured {hou_non_hr_runs} runs without the HR.")
print(f"The Yankees managed {nyy_non_hr_runs}.")
print()
print("This is what Hustle = -0.98 looks like in October:")
print("  - Can't steal bases to get into scoring position")
print("  - Can't take extra bases on singles and doubles")
print("  - Limited advancement on balls in dirt or fly balls")
print("  - When the HR dries up, the offense dies")
print()
print("The 2022 team showed defense can improve quickly.")
print("But Hustle (baserunning philosophy) was NEVER addressed under Fishman.")
print("That's not a talent problem — it's an analytics philosophy problem.")

In [ ]:
# === FIGURE 4: The 2022 Paradox ===
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Panel A: Value Composite component comparison (Yankees vs Astros 2022) ---
ax = axes[0]
components = ['Pressure', 'Hustle', 'Grit']
nyy_vals = [nyy_2022['pressure'], nyy_2022['hustle'], nyy_2022['grit']]
hou_vals = [hou_2022['pressure'], hou_2022['hustle'], hou_2022['grit']]

x = np.arange(len(components))
width = 0.35
bars_nyy = ax.bar(x - width/2, nyy_vals, width, label='Yankees', color='#003087', edgecolor='white')
bars_hou = ax.bar(x + width/2, hou_vals, width, label='Astros', color='#EB6E1F', edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=-0.5, color='#e74c3c', linewidth=1, linestyle='--', alpha=0.5, label='Imbalance threshold')
ax.set_xticks(x)
ax.set_xticklabels(components, fontweight='bold')
ax.set_ylabel('Z-Score')
ax.set_title('2022 ALCS: Value Composite Components\nYankees vs Astros', fontweight='bold')
ax.legend(fontsize=9)

# Annotate the Hustle hole
ax.annotate('FATAL\nFLAW', xy=(1 - width/2, nyy_2022['hustle']),
            xytext=(0.3, -1.8), fontsize=10, fontweight='bold', color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2),
            ha='center')

# --- Panel B: HR-dependency rank timeline ---
ax = axes[1]
hr_ranks = []
hr_pcts = []
for yr in range(2017, 2025):
    yr_s = value_composite_df[value_composite_df['Season'] == yr].copy()
    yr_s['HR_pct'] = yr_s['HR'] / yr_s['R'] * 100
    yr_s = yr_s.sort_values('HR_pct', ascending=False).reset_index(drop=True)
    nyy_r = yr_s[yr_s['Team'] == 'NYY'].index[0] + 1
    nyy_p = yr_s[yr_s['Team'] == 'NYY']['HR_pct'].values[0]
    hr_ranks.append(nyy_r)
    hr_pcts.append(nyy_p)

years = list(range(2017, 2025))
colors = ['#e74c3c' if r <= 3 else '#f39c12' if r <= 10 else '#95a5a6' for r in hr_ranks]
ax.bar(years, [31-r for r in hr_ranks], bottom=[r-1 for r in hr_ranks],
       color=colors, edgecolor='white', width=0.7)  # visual trick: bar height shows rank
# Actually just plot the rank directly
ax.clear()
ax.plot(years, hr_ranks, 'o-', color='#003087', linewidth=2.5, markersize=8, zorder=5)
ax.fill_between(years, hr_ranks, 30, alpha=0.1, color='#003087')
ax.axhspan(0.5, 3.5, alpha=0.15, color='#e74c3c', label='Top 3 (most HR-dependent)')
ax.set_ylim(30.5, 0.5)  # Invert: #1 at top
ax.set_xlabel('Season')
ax.set_ylabel('HR-Dependency Rank (1 = most dependent)')
ax.set_title('Yankees HR-Dependency Rank\n2017-2024', fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
for yr, rank in zip(years, hr_ranks):
    ax.annotate(f'#{rank}', (yr, rank), textcoords="offset points",
                xytext=(0, -15 if rank > 5 else 12), ha='center', fontsize=9, fontweight='bold')

# --- Panel C: First vs Second Half 2022 ---
ax = axes[2]
periods = ['First Half\n(Pre-ASB)', 'Second Half\n(Post-ASB)', 'ALCS\nvs Houston']
win_pcts = [64/92, 35/70, 0/4]
colors_c = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(periods, win_pcts, color=colors_c, edgecolor='white', width=0.6)
ax.axhline(y=0.500, color='black', linewidth=1, linestyle='--', alpha=0.5, label='.500')
ax.set_ylabel('Win %')
ax.set_title('2022 Yankees: Second-Half Regression\nFrom 117-Win Pace to ALCS Loss', fontweight='bold')
ax.set_ylim(0, 0.85)
ax.legend(fontsize=9)
for bar, pct in zip(bars, win_pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'.{int(pct*1000):03d}', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig("../outputs/figures/2022_paradox.png", dpi=150, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.show()
print("\nSaved: outputs/figures/2022_paradox.png")

---
## Part 6: The Extremes Trap — Gallo, IKF, and the Inability to Find Balance

Yankees analytics department couldn't find the middle ground between power and contact. They swung between extremes:
- **Joey Gallo** (traded for mid-2021): elite barrel rate (18.5%), elite exit velo — but a 35-40% K rate and zero ability to adjust. The analytics bet was that Yankee Stadium's short porch would unlock his pull power. It didn't.
- **Isiah Kiner-Falefa** (acquired before 2022): the over-correction. Contact-first, low K% — but 1.2% barrel rate and an OPS under .650. Putting the ball in play doesn't help when it's a weak grounder every time.

Gallo brought solid defense (positive OAA), so he wasn't zero-value. But the acquisition *thesis* — that his power profile would translate — failed completely. And IKF was proof the department couldn't course-correct without lurching to the opposite extreme.

The real problem: they kept buying **output** (barrel rate OR contact rate) without demanding the **process** (pitch recognition, chase rate, ability to adjust) that makes output sustainable. Meanwhile, contenders were filling their lineups with balanced hitters who could do both.

In [ ]:
# === Analysis 6A: Load expanded Statcast data for Gallo/IKF analysis ===
pitches_2021 = get_statcast_pitches(2021)
pitches_2022 = get_statcast_pitches(2022)
all_pitches_expanded = pd.concat([pitches_2021, pitches_2022, pitches_2023, pitches_2024], ignore_index=True)
print(f"Expanded pitch dataset: {len(all_pitches_expanded):,} pitches (2021-2024)")

# Player MLBAM IDs
GALLO_ID = 608336
IKF_ID = 643396

# Filter to relevant player-team combos
# Gallo with NYY: mid-2021 through mid-2022
gallo_all = all_pitches_expanded[all_pitches_expanded['batter'] == GALLO_ID]
gallo_nyy = gallo_all[
    ((gallo_all['home_team'] == 'NYY') | (gallo_all['away_team'] == 'NYY'))
    & (gallo_all['batter'] == GALLO_ID)
]
# IKF with NYY: 2022-2023
ikf_all = all_pitches_expanded[all_pitches_expanded['batter'] == IKF_ID]
ikf_nyy = ikf_all[
    ((ikf_all['home_team'] == 'NYY') | (ikf_all['away_team'] == 'NYY'))
]

print(f"Gallo NYY pitches: {len(gallo_nyy):,}")
print(f"IKF NYY pitches: {len(ikf_nyy):,}")

In [ ]:
# === Analysis 6B: The Extremes — Gallo vs IKF vs Contender Median ===

def compute_plate_metrics(pitches, label):
    """Compute key plate discipline and contact quality metrics from pitch-level data."""
    n_pitches = len(pitches)
    
    # Chase rate: swings on pitches outside the zone (zones 11-14)
    outside = pitches[pitches['zone'].isin([11, 12, 13, 14])]
    swings_outside = outside[outside['description'].isin([
        'swinging_strike', 'swinging_strike_blocked', 'foul', 'foul_tip',
        'foul_bunt', 'hit_into_play', 'hit_into_play_no_out', 'hit_into_play_score'
    ])]
    chase_rate = len(swings_outside) / len(outside) * 100 if len(outside) > 0 else 0
    
    # Whiff rate: swinging strikes / total swings
    all_swings = pitches[pitches['description'].isin([
        'swinging_strike', 'swinging_strike_blocked', 'foul', 'foul_tip',
        'foul_bunt', 'hit_into_play', 'hit_into_play_no_out', 'hit_into_play_score'
    ])]
    whiffs = pitches[pitches['description'].isin(['swinging_strike', 'swinging_strike_blocked'])]
    whiff_rate = len(whiffs) / len(all_swings) * 100 if len(all_swings) > 0 else 0
    
    # Batted ball quality
    batted = pitches[pitches['launch_speed'].notna()]
    avg_ev = batted['launch_speed'].mean() if len(batted) > 0 else 0
    avg_la = batted['launch_angle'].mean() if len(batted) > 0 else 0
    
    # Barrel rate (launch_speed_angle == 6 is "barrel")
    barrels = batted[batted['launch_speed_angle'] == 6]
    barrel_rate = len(barrels) / len(batted) * 100 if len(batted) > 0 else 0
    
    # K rate (approximate from pitch-level)
    strikeouts = pitches[pitches['events'] == 'strikeout']
    pas = pitches[pitches['events'].notna()]
    k_rate = len(strikeouts) / len(pas) * 100 if len(pas) > 0 else 0
    
    # BB rate
    walks = pitches[pitches['events'] == 'walk']
    bb_rate = len(walks) / len(pas) * 100 if len(pas) > 0 else 0
    
    # Ground ball rate
    gb = batted[batted['bb_type'] == 'ground_ball']
    gb_rate = len(gb) / len(batted) * 100 if len(batted) > 0 else 0
    
    return {
        'Label': label, 'Pitches': n_pitches, 'PAs': len(pas),
        'K%': k_rate, 'BB%': bb_rate, 'Chase%': chase_rate, 'Whiff%': whiff_rate,
        'Barrel%': barrel_rate, 'Avg EV': avg_ev, 'Avg LA': avg_la, 'GB%': gb_rate,
    }

# Compute metrics for Gallo and IKF
metrics = pd.DataFrame([
    compute_plate_metrics(gallo_nyy, 'Gallo (NYY)'),
    compute_plate_metrics(ikf_nyy, 'IKF (NYY)'),
])

# Compute contender median from FanGraphs (HOU/LAD/ATL qualified hitters, 2021-2023)
from fire_fishman.data.statcast import get_batting_stats
contender_teams = ['HOU', 'LAD', 'ATL']
contender_frames = []
for yr in [2021, 2022, 2023]:
    s = get_batting_stats(yr)
    q = s[(s['PA'] >= 200) & (s['Team'].isin(contender_teams))]
    contender_frames.append(q)
contender_all = pd.concat(contender_frames, ignore_index=True)

cont_median = {
    'Label': 'Contender median',
    'Pitches': '-', 'PAs': f"n={len(contender_all)}",
    'K%': contender_all['K%'].median() * 100,
    'BB%': contender_all['BB%'].median() * 100,
    'Chase%': '-',
    'Whiff%': '-',
    'Barrel%': contender_all['Barrel%'].median() * 100,
    'Avg EV': '-',
    'Avg LA': '-',
    'GB%': '-',
}

print("THE EXTREMES TRAP: GALLO vs IKF vs CONTENDER BASELINE")
print("=" * 85)
print(f"{'':>18} {'PAs':>6} {'K%':>7} {'BB%':>7} {'Chase%':>8} {'Whiff%':>8} {'Barrel%':>9} {'GB%':>6}")
print("-" * 85)
for _, row in metrics.iterrows():
    print(f"{row['Label']:>18} {int(row['PAs']):>6} {row['K%']:>6.1f}% {row['BB%']:>6.1f}% "
          f"{row['Chase%']:>7.1f}% {row['Whiff%']:>7.1f}% {row['Barrel%']:>8.1f}% {row['GB%']:>5.1f}%")
print(f"{'Contender median':>18} {'(n='+str(len(contender_all))+')':>6} "
      f"{cont_median['K%']:>6.1f}% {cont_median['BB%']:>6.1f}% "
      f"{'--':>8} {'--':>8} {cont_median['Barrel%']:>8.1f}% {'--':>6}")
print("-" * 85)
print()
print("Contender median = median qualified hitter on HOU/LAD/ATL (2021-2023)")
print()
print("Gallo: elite barrel rate, but K% nearly DOUBLE the contender baseline.")
print("IKF:   low K%, but barrel rate is 1/9th of what contenders expect from a hitter.")
print("Neither is close to what winning teams build around.")

# FanGraphs season stats for context
print()
print("FanGraphs SEASON STATS (wRC+, WAR)")
print("-" * 60)
for yr, name, team_note in [
    (2021, 'Gallo', 'TEX/NYY'), (2022, 'Gallo', 'NYY/LAD'),
    (2022, 'Kiner-Falefa', 'NYY'), (2023, 'Kiner-Falefa', 'NYY'),
]:
    stats = get_batting_stats(yr)
    p = stats[stats['Name'].str.contains(name, case=False, na=False)]
    if len(p) > 0:
        r = p.iloc[0]
        print(f"  {yr} {name:<18} ({team_note:<8}): wRC+ = {r['wRC+']:>3.0f}, WAR = {r['WAR']:>+4.1f}, "
              f"K% = {r['K%']*100:>5.1f}%, Barrel% = {r['Barrel%']*100:>5.1f}%")

print(f"\n  Contender median:                       wRC+ = {contender_all['wRC+'].median():>3.0f}, "
      f"WAR = {contender_all['WAR'].median():>+4.1f}, "
      f"K% = {contender_all['K%'].median()*100:>5.1f}%, "
      f"Barrel% = {contender_all['Barrel%'].median()*100:>5.1f}%")

In [ ]:
# === Analysis 6C: The Acquisition Thesis Failure ===

print("THE FISHMAN ACQUISITION PATTERN")
print("=" * 80)
print()
print("JOEY GALLO TRADE (July 2021)")
print("-" * 80)
print("  Analytics thesis: 'Elite barrel rate + Yankee Stadium short porch = HR explosion'")
print("  Pre-trade (TEX 2021): .223/.379/.490, 25 HR, 34.6% K, 18.5% barrel rate")
print("  With NYY (2021-22):  .159/.291/.368, 25 HR, 39.3% K — K% went UP")
print("  The problem: barrel rate means nothing if you can't recognize the slider.")
print("  Gallo's chase rate on breaking balls with NYY was a high-risk profile.")
print()
print("IKF TRADE (March 2022)")
print("-" * 80)
print("  Analytics thesis: 'After Gallo, get a contact guy who puts it in play'")
print("  With NYY (2022-23):  .265/.310/.336, 1.2-3.1% barrel rate, 84-81 wRC+")
print("  The problem: contact without quality is just outs with extra steps.")
print("  IKF's ground ball rate was elite-bad. Weak contact = double plays.")
print()

# What contenders were building: the balanced hitter count
print("THE BALANCED HITTER GAP: 'SWEET SPOT' PLAYERS (2021-2023)")
print("=" * 80)
print("Sweet spot = K% < 25% AND Barrel% > 6% AND BB% > 8%")
print("(Can hit for power, controls the zone, walks)")
print("-" * 80)

from fire_fishman.data.statcast import get_batting_stats

teams_of_interest = {
    'NYY': 'Yankees',
    'HOU': 'Astros',
    'LAD': 'Dodgers', 
    'ATL': 'Braves',
}

for yr in [2021, 2022, 2023]:
    stats = get_batting_stats(yr)
    # Filter to qualified-ish batters (>200 PA)
    qualified = stats[stats['PA'] >= 200].copy()
    sweet = qualified[
        (qualified['K%'] < 0.25) & 
        (qualified['Barrel%'] > 0.06) & 
        (qualified['BB%'] > 0.08)
    ]
    
    print(f"\n{yr}:")
    for abbr, name in teams_of_interest.items():
        team_sweet = sweet[sweet['Team'] == abbr]
        team_total = qualified[qualified['Team'] == abbr]
        count = len(team_sweet)
        total = len(team_total)
        names = ', '.join(team_sweet['Name'].values[:4])
        print(f"  {name:<10}: {count}/{total} qualified batters in sweet spot"
              + (f" ({names})" if names else ""))

print()
print("Contenders found balanced hitters. The Yankees kept oscillating between")
print("all-or-nothing sluggers and slap hitters — never the complete player.")

In [ ]:
# === FIGURE 5: The Extremes Trap ===
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Get all qualified batters for scatter plot (2021-2023)
from fire_fishman.data.statcast import get_batting_stats
all_batters = []
for yr in [2021, 2022, 2023]:
    s = get_batting_stats(yr)
    s = s[s['PA'] >= 200].copy()
    s['Year'] = yr
    all_batters.append(s)
all_batters = pd.concat(all_batters, ignore_index=True)

# --- Panel A: Barrel% vs K% scatter with Gallo/IKF as extremes ---
ax = axes[0]
ax.scatter(all_batters['K%'] * 100, all_batters['Barrel%'] * 100, 
           alpha=0.15, color='#95a5a6', s=20, zorder=1)

# Sweet spot zone
from matplotlib.patches import Rectangle
sweet_rect = Rectangle((5, 6), 20, 20, linewidth=2, edgecolor='#2ecc71', 
                         facecolor='#2ecc71', alpha=0.1, linestyle='--', zorder=2)
ax.add_patch(sweet_rect)
ax.text(14, 24, 'SWEET\nSPOT', fontsize=9, fontweight='bold', color='#2ecc71',
        ha='center', va='center', zorder=3)

# Highlight Gallo
gallo_stats = all_batters[all_batters['Name'].str.contains('Gallo', na=False)]
ax.scatter(gallo_stats['K%'] * 100, gallo_stats['Barrel%'] * 100,
           color='#e74c3c', s=100, zorder=5, edgecolors='black', linewidth=1)
for _, r in gallo_stats.iterrows():
    ax.annotate(f"Gallo\n'{int(r['Year'])-2000}", (r['K%']*100, r['Barrel%']*100),
                textcoords="offset points", xytext=(12, -5), fontsize=8, fontweight='bold',
                color='#e74c3c')

# Highlight IKF
ikf_stats = all_batters[all_batters['Name'].str.contains('Kiner-Falefa', na=False)]
ax.scatter(ikf_stats['K%'] * 100, ikf_stats['Barrel%'] * 100,
           color='#3498db', s=100, zorder=5, edgecolors='black', linewidth=1)
for _, r in ikf_stats.iterrows():
    ax.annotate(f"IKF\n'{int(r['Year'])-2000}", (r['K%']*100, r['Barrel%']*100),
                textcoords="offset points", xytext=(-30, 8), fontsize=8, fontweight='bold',
                color='#3498db')

# Contender median marker
cont_k = contender_all['K%'].median() * 100
cont_barrel = contender_all['Barrel%'].median() * 100
ax.scatter([cont_k], [cont_barrel], color='#2ecc71', s=150, zorder=6, 
           edgecolors='black', linewidth=2, marker='*')
ax.annotate('Contender\nmedian', (cont_k, cont_barrel),
            textcoords="offset points", xytext=(-35, -20), fontsize=8, fontweight='bold',
            color='#2ecc71')

ax.set_xlabel('K%')
ax.set_ylabel('Barrel%')
ax.set_title('Barrel% vs K%: All Qualified Batters\n2021-2023', fontweight='bold')
ax.set_xlim(5, 45)
ax.set_ylim(0, 26)

# --- Panel B: Sweet spot player count by team ---
ax = axes[1]
teams_plot = {'NYY': '#003087', 'HOU': '#EB6E1F', 'LAD': '#005A9C', 'ATL': '#CE1141'}
x_pos = np.arange(3)
width = 0.2
for i, (abbr, color) in enumerate(teams_plot.items()):
    counts = []
    for yr in [2021, 2022, 2023]:
        s = get_batting_stats(yr)
        q = s[(s['PA'] >= 200) & (s['Team'] == abbr)]
        sweet = q[(q['K%'] < 0.25) & (q['Barrel%'] > 0.06) & (q['BB%'] > 0.08)]
        counts.append(len(sweet))
    ax.bar(x_pos + i * width, counts, width, label=abbr, color=color, edgecolor='white')

ax.set_xticks(x_pos + 1.5 * width)
ax.set_xticklabels(['2021', '2022', '2023'])
ax.set_ylabel('# of "Sweet Spot" Batters (PA >= 200)')
ax.set_title('"Complete Hitters" Per Team\nK% < 25%, Barrel% > 6%, BB% > 8%', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, max(ax.get_ylim()[1], 6))

# --- Panel C: Gallo K% trajectory showing he never adjusted ---
ax = axes[2]
import calendar
gallo_nyy_monthly = gallo_nyy.copy()
gallo_nyy_monthly['month'] = pd.to_datetime(gallo_nyy_monthly['game_date']).dt.month
gallo_nyy_monthly['year'] = pd.to_datetime(gallo_nyy_monthly['game_date']).dt.year

monthly_data = []
for (yr, mo), group in gallo_nyy_monthly.groupby(['year', 'month']):
    pas = group[group['events'].notna()]
    ks = pas[pas['events'] == 'strikeout']
    if len(pas) >= 20:
        monthly_data.append({
            'period': f"{calendar.month_abbr[mo]}\n'{yr-2000}",
            'K%': len(ks) / len(pas) * 100,
            'PA': len(pas)
        })

if monthly_data:
    mdf = pd.DataFrame(monthly_data)
    colors_m = ['#f39c12' if k < 35 else '#e74c3c' for k in mdf['K%']]
    ax.bar(range(len(mdf)), mdf['K%'], color=colors_m, edgecolor='white', width=0.7)
    ax.set_xticks(range(len(mdf)))
    ax.set_xticklabels(mdf['period'], fontsize=8)
    ax.axhline(y=cont_k, color='#2ecc71', linewidth=2, linestyle='--', 
               label=f'Contender median K% ({cont_k:.0f}%)', alpha=0.7)
    ax.axhline(y=35, color='#e74c3c', linewidth=1, linestyle='--', alpha=0.5, label='35% threshold')
    ax.set_ylabel('K%')
    ax.set_title("Gallo's Monthly K% with Yankees\nHe Never Adjusted", fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.set_ylim(0, 55)

plt.tight_layout()
plt.savefig("../outputs/figures/extremes_trap.png", dpi=150, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.show()
print("\nSaved: outputs/figures/extremes_trap.png")

---
## Part 3: The Yankees analytics leadership Scorecard

Quantifying the total estimated impact from these areas.

In [ ]:
# === Analytics Decision Scorecard (Updated) ===

print("=" * 70)
print("ANALYTICS DECISION SCORECARD")
print("=" * 70)
print()

# --- Mixed Evidence: Short Porch ---
print("MIXED EVIDENCE: Short porch LHH advantage vs overall RHH park factor")
print("-" * 70)
# Use all PA at Yankee Stadium (home + away batters) to match HR numerator
ys_all_pa = pa_events[pa_events["home_team"] == "NYY"]
ys_lhh_pa = len(ys_all_pa[ys_all_pa["stand"] == "L"])
ys_rhh_pa = len(ys_all_pa[ys_all_pa["stand"] == "R"])
lhh_rf_rate = lhh_pull_count / ys_lhh_pa
rhh_rf_rate = rhh_oppo_count / ys_rhh_pa
print(f"  LHH pull HRs to RF: {lhh_pull_count} ({lhh_rf_rate:.2%}/PA)")
print(f"  RHH oppo HRs to RF: {rhh_oppo_count} ({rhh_rf_rate:.2%}/PA)")
print(f"  LHH are {lhh_rf_rate/rhh_rf_rate:.1f}x more productive to the short porch")
print(f"  Yankees RHH% at home: {nyy_rhh:.1%} vs league {league_avg_rhh:.1%}")
print(f"  Pitchers faced the same handedness 7 batters in a row — one gameplan")
print(f"  NOTE: Overall HR park factor favors RHH (see Analysis 1D)")
print(f"  The LHH advantage is specific to RF pull HRs, not overall production")
print()

# --- Decision Area #2: Baserunning ---
print("DECISION AREA #2: 'Limited emphasis on baserunning value'")
print("-" * 70)
full_bsr = nyy_stats[nyy_stats["Season"].between(2018, 2024)]["BsR"].sum()
full_ubr = nyy_stats[nyy_stats["Season"].between(2018, 2024)]["UBR"].sum()
print(f"  BsR trajectory: 7th (2017) -> 24th (2022) -> 30th (2024)")
print(f"  Total BsR (2018-2024): {full_bsr:+.1f} runs = {abs(full_bsr)/10:.1f} wins lost")
print(f"  Total UBR (2018-2024): {full_ubr:+.1f} runs (can't run the bases)")
print(f"  #1 most HR-dependent team: 2018, 2022, 2023")
print()

# --- Decision Area #3: Defense ---
print("DECISION AREA #3: 'Just hit homers, defense doesn't matter'")
print("-" * 70)
bad_def = all_fielding[
    (all_fielding["Team"] == "Yankees") & (all_fielding["Season"].between(2018, 2021))
]
total_def = bad_def["Def"].sum()
total_oaa = bad_def["OAA"].sum()
print(f"  OAA (2018-2021): {total_oaa:+.0f} (2nd worst in MLB)")
print(f"  Total Def (2018-2021): {total_def:+.1f} runs = {abs(total_def)/10:.1f} wins lost")
print(f"  Then showed defense was available: 2022 = #1 DRS (+124)")
print(f"  They CHOSE to be bad at defense for 4 years")
print()

# --- The October Problem ---
print("POSTSEASON RISK: limited secondary value and offensive diversity")
print("-" * 70)
print(f"  2018-2024: Lost ALCS/WS to HOU (3x), BOS (2x), TB, LAD")
print(f"  Every opponent: elite defense + speed + lineup diversity")
print(f"  Yankees: wait for HR against elite pitching = no runs")
print(f"  Only Stanton consistently shows up in October")
print(f"  When the HR doesn't come, there's literally nothing else")
print()

# --- Combined ---
print("=" * 70)
print("COMBINED ESTIMATED IMPACT:")
print(f"  Baserunning (2018-2024): {abs(full_bsr)/10:.1f} wins lost")
print(f"  Defense (2018-2021):     {abs(total_def)/10:.1f} wins lost")
print(f"  Handedness mismatch:     ~3-4 wins lost (est.)")
print(f"  ────────────────────────────────────")
total_wins = abs(full_bsr)/10 + abs(total_def)/10 + 3.5
print(f"  TOTAL:                   ~{total_wins:.0f} wins left on the table")
print()
print("That doesn't include the prospect development failures,")
print("the postseason outcomes, or the estimated cost of building")
print("a one-dimensional roster in the most versatile park in baseball.")
print()
print("The adjustment was available — the 2022 team showed it.")
print("They just had an analytics department that didn't believe")
print("in defense, baserunning, lineup diversity, or aggressiveness.")
print("They believed in one thing: the three-run homer.")
print("And when it doesn't come, you go home in October.")
print("=" * 70)